<a href="https://colab.research.google.com/github/ShahSayem/Alzheimer-Disease-Detection/blob/main/2B-OASIS_Alzheimer_ViT_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
ninadaithal_imagesoasis_path = kagglehub.dataset_download('ninadaithal/imagesoasis')

print('Data source import complete.')


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models, transforms # models will be used for ViT
from PIL import Image
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score
import time # For timing epochs

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# Data Collection
base_path = '/kaggle/input/imagesoasis/Data/'

non_demented_paths = []
very_mild_demented_paths = []
mild_demented_paths = []
moderate_demented_paths = []

paths_dict = {
    "Non Demented": non_demented_paths,
    "Very mild Dementia": very_mild_demented_paths,
    "Mild Dementia": mild_demented_paths,
    "Moderate Dementia": moderate_demented_paths
}

for class_name, path_list in paths_dict.items():
    class_dir = os.path.join(base_path, class_name)
    if os.path.isdir(class_dir):
        for filename in os.listdir(class_dir):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                path_list.append(os.path.join(class_dir, filename))
    else:
        print(f"Warning: Directory not found {class_dir}")

print(f"Non Demented: {len(non_demented_paths)}")
print(f"Mild Demented: {len(mild_demented_paths)}")
print(f"Moderate Demented: {len(moderate_demented_paths)}")
print(f"Very Mild Demented: {len(very_mild_demented_paths)}")

# Create DataFrame and Label Encoding
all_image_paths = []
all_labels_str = []

label_map = {
    "Non Demented": 0,
    "Mild Dementia": 1,
    "Moderate Dementia": 2,
    "Very mild Dementia": 3
}
idx_to_class = {v: k for k, v in label_map.items()}

for label_str, paths in paths_dict.items():
    if label_str in label_map:
        label_int = label_map[label_str]
        for path in paths:
            all_image_paths.append(path)
            all_labels_str.append(label_str)
    else:
        print(f"Warning: Class name '{label_str}' from directory structure not in label_map.")

df = pd.DataFrame({
    'image_path': all_image_paths,
    'label_str': all_labels_str
})
df['label_int'] = df['label_str'].map(label_map)
df.dropna(subset=['label_int'], inplace=True)
df['label_int'] = df['label_int'].astype(int)

print("\nDataFrame Head:")
print(df.head())
print("\nClass Distribution in DataFrame:")
print(df['label_str'].value_counts())

Non Demented: 67222
Mild Demented: 5002
Moderate Demented: 488
Very Mild Demented: 13725

DataFrame Head:
                                          image_path     label_str  label_int
0  /kaggle/input/imagesoasis/Data/Non Demented/OA...  Non Demented          0
1  /kaggle/input/imagesoasis/Data/Non Demented/OA...  Non Demented          0
2  /kaggle/input/imagesoasis/Data/Non Demented/OA...  Non Demented          0
3  /kaggle/input/imagesoasis/Data/Non Demented/OA...  Non Demented          0
4  /kaggle/input/imagesoasis/Data/Non Demented/OA...  Non Demented          0

Class Distribution in DataFrame:
label_str
Non Demented          67222
Very mild Dementia    13725
Mild Dementia          5002
Moderate Dementia       488
Name: count, dtype: int64


In [ ]:
# PyTorch Custom Dataset
class AlzheimerDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx]['image_path']
        label = self.dataframe.iloc[idx]['label_int']
        try:
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            print(f"Error: Image not found at {img_path}")
            return torch.randn(3, 224, 224), torch.tensor(-1)
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            return torch.randn(3, 224, 224), torch.tensor(-1)

        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

# Image Transformations for ViT
# ViT models expect a fixed input size (e.g., 224x224 for vit_b_16)
try:
    vit_weights = models.ViT_B_16_Weights.IMAGENET1K_V1
    vit_preprocess = vit_weights.transforms()
    IMG_SIZE = 224 # vit_b_16 default with these weights is 224
    print(f"Using official ViT preprocessing. Expected image size: {IMG_SIZE}x{IMG_SIZE}")
except AttributeError: # Fallback for older torchvision or if specific weights object isn't found
    print("ViT_B_16_Weights.IMAGENET1K_V1 not found, using manual transforms. Ensure torchvision is up to date.")
    IMG_SIZE = 224 # Common ViT input size
    vit_preprocess = transforms.Compose([
        transforms.Resize(IMG_SIZE), # Or transforms.Resize((IMG_SIZE, IMG_SIZE))
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # Standard ImageNet stats
    ])


data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)), # Crop to ViT's expected input size
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        # The following are essential and often part of the official preprocess
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # Using standard ImageNet stats
    ]),
    'val': vit_preprocess,
    'test': vit_preprocess,
}


# Splitting Data
if df.empty:
    print("DataFrame is empty. Cannot proceed with splitting and training.")
else:
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_int'])
    train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df['label_int'])

    print(f"Training samples: {len(train_df)}")
    print(f"Validation samples: {len(val_df)}")
    print(f"Test samples: {len(test_df)}")

    train_dataset = AlzheimerDataset(train_df, transform=data_transforms['train'])
    val_dataset = AlzheimerDataset(val_df, transform=data_transforms['val'])
    test_dataset = AlzheimerDataset(test_df, transform=data_transforms['test']) # Use 'test' transform

    # ViTs can be memory intensive, consider a smaller batch size if needed
    BATCH_SIZE = 16
    num_workers_val = os.cpu_count()
    if num_workers_val is None: # Handle case where os.cpu_count() might be None
        num_workers_val = 1
    else:
        num_workers_val = min(4, num_workers_val)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers_val)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers_val)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers_val)

    dataloaders = {'train': train_loader, 'val': val_loader}
    dataset_sizes = {'train': len(train_dataset), 'val': len(val_dataset)}

Using official ViT preprocessing. Expected image size: 224x224
Training samples: 55319
Validation samples: 13830
Test samples: 17288


In [ ]:
# ViT Model Setup
num_classes = len(df['label_int'].unique()) if not df.empty else 0
print(f"Number of classes: {num_classes}")

if num_classes > 0:
    try:
        # Use the same weights object as for transforms for consistency
        model_vit = models.vit_b_16(weights=vit_weights)
    except NameError: # If vit_weights was not defined due to fallback
         print("Using older torchvision pretrained=True method for ViT_B_16.")
         model_vit = models.vit_b_16(pretrained=True) # For older torchvision
    except TypeError: # Another fallback for older torchvision if `weights` arg is not recognized
         print("Using older torchvision pretrained=True method for ViT_B_16 (TypeError).")
         model_vit = models.vit_b_16(pretrained=True)


    # ViT's classification head is model.heads, which is an nn.Sequential
    # The actual linear layer is typically model.heads.head
    # For vit_b_16, hidden_dim is 768
    hidden_dim = model_vit.heads.head.in_features # Get the number of input features
    model_vit.heads.head = nn.Linear(hidden_dim, num_classes) # Replace the head

    model_vit = model_vit.to(device)
else:
    model_vit = None # Define model_vit as None if no classes
    print("Model not loaded as num_classes is 0 or DataFrame is empty.")

# Loss Function and Optimizer
if model_vit: # Only define if model is loaded
    criterion = nn.CrossEntropyLoss()
    # ViTs can be sensitive to learning rates. AdamW is often preferred.
    optimizer_ft = optim.AdamW(model_vit.parameters(), lr=1e-5, weight_decay=1e-4) # Smaller LR for fine-tuning ViT

    # Decay LR by a factor of 0.1 every 7 epochs (can be adjusted)
    exp_lr_scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=7, gamma=0.1)
else:
    criterion, optimizer_ft, exp_lr_scheduler = None, None, None

Number of classes: 4


Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth
100%|██████████| 330M/330M [00:04<00:00, 76.1MB/s] 


In [ ]:
# Training Function
def train_model(model, criterion, optimizer, scheduler, num_epochs=25):
    since = time.time()
    best_model_wts = model.state_dict()
    best_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()
            running_loss = 0.0
            running_corrects = 0
            progress_bar = tqdm(dataloaders[phase], desc=f"{phase.capitalize()} Epoch {epoch+1}")
            for inputs, labels in progress_bar:
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                progress_bar.set_postfix(loss=loss.item(), acc=torch.sum(preds == labels.data).item()/inputs.size(0))
            if phase == 'train' and scheduler is not None: # Check if scheduler is defined
                scheduler.step()
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.item())
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.item())
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = model.state_dict()
                torch.save(model.state_dict(), 'best_vit_model.pth') # Save best ViT model
                print(f"New best validation accuracy: {best_acc:.4f}, model saved.")
        print()
    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')
    model.load_state_dict(best_model_wts)
    return model, history

In [ ]:
# Start Training (if data is loaded and model is configured)
if not df.empty and model_vit and criterion:
    print("\nStarting ViT model training...")
    # ViTs might need more epochs or different learning rate schedules.
    # Adjust num_epochs as needed.
    model_vit_trained, history = train_model(model_vit, criterion, optimizer_ft, exp_lr_scheduler, num_epochs=15)
else:
    print("Skipping training as DataFrame is empty, model not loaded, or criterion not set.")
    model_vit_trained = None
    history = None

# Plot Training History (same as before)
if history:
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.title('ViT Accuracy over Epochs')

    plt.subplot(1, 2, 2)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('ViT Loss over Epochs')
    plt.tight_layout()
    plt.show()


Starting ViT model training...
Epoch 1/15
----------


Train Epoch 1: 100%|██████████| 3458/3458 [18:43<00:00,  3.08it/s, acc=1, loss=0.0333]    


train Loss: 0.2680 Acc: 0.8914


Val Epoch 1: 100%|██████████| 865/865 [01:27<00:00,  9.91it/s, acc=0.833, loss=0.608] 


val Loss: 0.5356 Acc: 0.8269
New best validation accuracy: 0.8269, model saved.

Epoch 2/15
----------


...

Train Epoch 11: 100%|██████████| 3458/3458 [18:44<00:00,  3.08it/s, acc=1, loss=2.7e-5]    


train Loss: 0.0002 Acc: 0.9999


Val Epoch 11: 100%|██████████| 865/865 [01:27<00:00,  9.89it/s, acc=1, loss=0.104]     


val Loss: 0.6398 Acc: 0.8651

Epoch 12/15
----------


Train Epoch 12:  17%|█▋        | 573/3458 [03:06<15:39,  3.07it/s, acc=1, loss=3.17e-5] 

In [ ]:
# Evaluation on Test Set
def evaluate_model(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Evaluating Test Set"):
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return np.array(all_labels), np.array(all_preds)

if model_vit_trained:
    print("\nEvaluating ViT model on the test set...")
    try:
        model_vit_trained.load_state_dict(torch.load('best_vit_model.pth'))
        print("Loaded best ViT model weights for evaluation.")
    except FileNotFoundError:
        print("Warning: 'best_vit_model.pth' not found. Evaluating with current ViT model weights.")

    y_true_test, y_pred_test = evaluate_model(model_vit_trained, test_loader)
    test_accuracy = accuracy_score(y_true_test, y_pred_test)
    print(f"\nViT Test Accuracy: {test_accuracy:.4f}")
    print("\nViT Classification Report (Test Set):")
    target_names_report = [idx_to_class[i] for i in sorted(idx_to_class.keys())]
    print(classification_report(y_true_test, y_pred_test, target_names=target_names_report, zero_division=0))
    print("\nViT Confusion Matrix (Test Set):")
    cm = confusion_matrix(y_true_test, y_pred_test)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names_report)
    disp.plot(cmap=plt.cm.Blues)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("Skipping ViT model evaluation as the model was not trained.")